In [30]:
import pandas as pd
from pathlib import Path

ruta_csv = Path(r"../Data/sidpol_consolidado_final_limpio_v5_comisarias_2020_por_nombre.csv")

df = pd.read_csv(
    ruta_csv,
    encoding="utf-8-sig",
    low_memory=False
)

In [31]:
df.head()

,anio,mes,fecha,anio_mes,trimestre,ubigeo,departamento,provincia,distrito,poblacion,...,n_policias_administrativo_ref_2025,pct_policias_operativo_ref_2025,anio_fuente_policias,mes_fuente_policias,fuente_policias,n_comisarias_tipo_A_ref_2020,n_comisarias_tipo_B_ref_2020,n_comisarias_tipo_C_ref_2020,n_comisarias_tipo_D_ref_2020,n_comisarias_tipo_E_ref_2020
0,2018,1,2018-01-01,2018-01,1,10202,AMAZONAS,BAGUA,ARAMANGO,10422,...,0,100.0,2025,4,MININTER - EFECTIVOS POLICIALES PNP ABRIL 2025,0,0,0,0,2
1,2018,1,2018-01-01,2018-01,1,10202,AMAZONAS,BAGUA,ARAMANGO,10422,...,0,100.0,2025,4,MININTER - EFECTIVOS POLICIALES PNP ABRIL 2025,0,0,0,0,2
2,2018,1,2018-01-01,2018-01,1,10201,AMAZONAS,BAGUA,BAGUA,29096,...,16,86.4,2025,4,MININTER - EFECTIVOS POLICIALES PNP ABRIL 2025,0,1,0,0,0
3,2018,1,2018-01-01,2018-01,1,10201,AMAZONAS,BAGUA,BAGUA,29096,...,16,86.4,2025,4,MININTER - EFECTIVOS POLICIALES PNP ABRIL 2025,0,1,0,0,0
4,2018,1,2018-01-01,2018-01,1,10201,AMAZONAS,BAGUA,BAGUA,29096,...,16,86.4,2025,4,MININTER - EFECTIVOS POLICIALES PNP ABRIL 2025,0,1,0,0,0


In [32]:
resumen_columnas_v5 = pd.DataFrame({
    "columna": df.columns,
    "tipo_dato": df.dtypes.astype(str).values,
    "cantidad_nulos": df.isnull().sum().values,
    "porcentaje_nulos": (df.isnull().mean() * 100).round(2).values
})
carpeta_salida = Path(r"../Data/salida_revision")
carpeta_salida.mkdir(parents=True, exist_ok=True)
ruta_salida = carpeta_salida / "resumen_columnas_v5.csv"
resumen_columnas_v5.to_csv(ruta_salida, index=False, encoding="utf-8-sig")

# Entrega 3: Preprocesamiento para modelado

## Objetivo

Este notebook realiza el preprocesamiento necesario para dejar la base lista para una etapa posterior de modelado.

El objetivo del futuro modelo será predecir la variable:

- cantidad: número de denuncias registradas.

En esta etapa no se entrenan modelos. Solo se prepara el dataset mediante limpieza, transformación de tipos de datos, creación de variables temporales, control de columnas con fuga de información y separación entre entrenamiento y prueba.

In [33]:
import numpy as np

In [34]:
resumen_inicial = pd.DataFrame({
    "columna": df.columns,
    "tipo_dato": df.dtypes.astype(str).values,
    "nulos": df.isna().sum().values,
    "porcentaje_nulos": (df.isna().mean() * 100).round(2).values,
    "valores_unicos": df.nunique(dropna=True).values
})

resumen_inicial

,columna,tipo_dato,nulos,porcentaje_nulos,valores_unicos
0,anio,int64,0,0.00,8
1,mes,int64,0,0.00,12
2,fecha,str,0,0.00,93
3,anio_mes,str,0,0.00,93
4,trimestre,int64,0,0.00,4
5,ubigeo,int64,0,0.00,1833
6,departamento,str,0,0.00,26
7,provincia,str,0,0.00,195
8,distrito,str,0,0.00,1682
9,poblacion,int64,0,0.00,9061


En esta sección se prepara la base de datos para una etapa posterior de modelado predictivo.

El objetivo futuro será predecir la variable cantidad, que representa el número de denuncias registradas.

In [35]:
df_pre = df.copy()

df_pre["fecha"] = pd.to_datetime(df_pre["fecha"], errors="coerce")

df_pre["ubigeo"] = (
    df_pre["ubigeo"]
    .astype(str)
    .str.replace(r"\.0$", "", regex=True)
    .str.zfill(6)
)

df_pre["cantidad"] = pd.to_numeric(df_pre["cantidad"], errors="coerce")

columnas_texto = df_pre.select_dtypes(include=["object", "string"]).columns.tolist()

for col in columnas_texto:
    df_pre[col] = (
        df_pre[col]
        .astype("string")
        .str.strip()
        .str.replace(r"\s+", " ", regex=True)
    )

print("Tamaño:", df_pre.shape)
df_pre.head()

Tamaño: (772656, 56)


,anio,mes,fecha,anio_mes,trimestre,ubigeo,departamento,provincia,distrito,poblacion,...,n_policias_administrativo_ref_2025,pct_policias_operativo_ref_2025,anio_fuente_policias,mes_fuente_policias,fuente_policias,n_comisarias_tipo_A_ref_2020,n_comisarias_tipo_B_ref_2020,n_comisarias_tipo_C_ref_2020,n_comisarias_tipo_D_ref_2020,n_comisarias_tipo_E_ref_2020
0,2018,1,2018-01-01,2018-01,1,010202,AMAZONAS,BAGUA,ARAMANGO,10422,...,0,100.0,2025,4,MININTER - EFECTIVOS POLICIALES PNP ABRIL 2025,0,0,0,0,2
1,2018,1,2018-01-01,2018-01,1,010202,AMAZONAS,BAGUA,ARAMANGO,10422,...,0,100.0,2025,4,MININTER - EFECTIVOS POLICIALES PNP ABRIL 2025,0,0,0,0,2
2,2018,1,2018-01-01,2018-01,1,010201,AMAZONAS,BAGUA,BAGUA,29096,...,16,86.4,2025,4,MININTER - EFECTIVOS POLICIALES PNP ABRIL 2025,0,1,0,0,0
3,2018,1,2018-01-01,2018-01,1,010201,AMAZONAS,BAGUA,BAGUA,29096,...,16,86.4,2025,4,MININTER - EFECTIVOS POLICIALES PNP ABRIL 2025,0,1,0,0,0
4,2018,1,2018-01-01,2018-01,1,010201,AMAZONAS,BAGUA,BAGUA,29096,...,16,86.4,2025,4,MININTER - EFECTIVOS POLICIALES PNP ABRIL 2025,0,1,0,0,0


In [36]:
validacion_inicial = pd.DataFrame({
    "validacion": [
        "fechas_nulas",
        "cantidad_nula",
        "cantidad_negativa"
    ],
    "valor": [
        df_pre["fecha"].isna().sum(),
        df_pre["cantidad"].isna().sum(),
        (df_pre["cantidad"] < 0).sum()
    ]
})

validacion_inicial

,validacion,valor
0,fechas_nulas,0
1,cantidad_nula,0
2,cantidad_negativa,0


In [37]:
filas_antes = df_pre.shape[0]

df_pre = df_pre.dropna(subset=["fecha", "cantidad"])
df_pre = df_pre[df_pre["cantidad"] >= 0]

filas_despues = df_pre.shape[0]

pd.DataFrame({
    "elemento": ["filas_antes", "filas_despues", "filas_eliminadas"],
    "valor": [filas_antes, filas_despues, filas_antes - filas_despues]
})

,elemento,valor
0,filas_antes,772656
1,filas_despues,772656
2,filas_eliminadas,0


In [38]:
df_pre["anio"] = df_pre["fecha"].dt.year
df_pre["mes"] = df_pre["fecha"].dt.month
df_pre["trimestre"] = df_pre["fecha"].dt.quarter
df_pre["periodo"] = df_pre["fecha"].dt.to_period("M").astype(str)

df_pre["mes_sin"] = np.sin(2 * np.pi * df_pre["mes"] / 12)
df_pre["mes_cos"] = np.cos(2 * np.pi * df_pre["mes"] / 12)

df_pre[["fecha", "anio", "mes", "trimestre", "periodo", "mes_sin", "mes_cos"]].head()

,fecha,anio,mes,trimestre,periodo,mes_sin,mes_cos
0,2018-01-01,2018,1,1,2018-01,0.5,0.866025
1,2018-01-01,2018,1,1,2018-01,0.5,0.866025
2,2018-01-01,2018,1,1,2018-01,0.5,0.866025
3,2018-01-01,2018,1,1,2018-01,0.5,0.866025
4,2018-01-01,2018,1,1,2018-01,0.5,0.866025


In [39]:
columnas_fuga = [
    "tasa_denuncias_100k",
    "denuncias_por_comisaria_ref_2020",
    "denuncias_por_policia_ref_2025",
    "cantidad_departamento_delitos",
    "cantidad_departamento_misma_categoria",
    "participacion_en_delitos_departamento_pct",
    "participacion_misma_categoria_departamento_pct",
    "tasa_departamento_delitos_100k_referencial"
]

columnas_metadata = [
    "fecha",
    "anio_mes",
    "periodo",
    "anio_fuente_comisarias",
    "fuente_comisarias",
    "anio_fuente_policias",
    "mes_fuente_policias",
    "fuente_policias"
]

columnas_objetivo = [
    "cantidad"
]

columnas_excluir = columnas_fuga + columnas_metadata + columnas_objetivo

columnas_excluir = [
    col for col in columnas_excluir
    if col in df_pre.columns
]

columnas_excluir

['tasa_denuncias_100k',
 'denuncias_por_comisaria_ref_2020',
 'denuncias_por_policia_ref_2025',
 'cantidad_departamento_delitos',
 'cantidad_departamento_misma_categoria',
 'participacion_en_delitos_departamento_pct',
 'participacion_misma_categoria_departamento_pct',
 'tasa_departamento_delitos_100k_referencial',
 'fecha',
 'anio_mes',
 'periodo',
 'anio_fuente_comisarias',
 'fuente_comisarias',
 'anio_fuente_policias',
 'mes_fuente_policias',
 'fuente_policias',
 'cantidad']

In [40]:
columnas_constantes = []

for col in df_pre.columns:
    if df_pre[col].nunique(dropna=True) <= 1:
        columnas_constantes.append(col)

columnas_constantes

['anio_fuente_comisarias', 'anio_fuente_policias', 'mes_fuente_policias']

In [41]:
columnas_no_modelo = list(set(columnas_excluir + columnas_constantes))

columnas_predictoras = [
    col for col in df_pre.columns
    if col not in columnas_no_modelo
]

df_modelo = df_pre[columnas_predictoras + ["cantidad"]].copy()

print("Tamaño de df_pre:", df_pre.shape)
print("Tamaño de df_modelo:", df_modelo.shape)

df_modelo.head()

Tamaño de df_pre: (772656, 59)
Tamaño de df_modelo: (772656, 43)


,anio,mes,trimestre,ubigeo,departamento,provincia,distrito,poblacion,n_comisarias_ref_2020,comisarias_por_100k_ref_2020,...,n_policias_administrativo_ref_2025,pct_policias_operativo_ref_2025,n_comisarias_tipo_A_ref_2020,n_comisarias_tipo_B_ref_2020,n_comisarias_tipo_C_ref_2020,n_comisarias_tipo_D_ref_2020,n_comisarias_tipo_E_ref_2020,mes_sin,mes_cos,cantidad
0,2018,1,1,010202,AMAZONAS,BAGUA,ARAMANGO,10422,2,19.1902,...,0,100.0,0,0,0,0,2,0.5,0.866025,3
1,2018,1,1,010202,AMAZONAS,BAGUA,ARAMANGO,10422,2,19.1902,...,0,100.0,0,0,0,0,2,0.5,0.866025,3
2,2018,1,1,010201,AMAZONAS,BAGUA,BAGUA,29096,1,3.4369,...,16,86.4,0,1,0,0,0,0.5,0.866025,2
3,2018,1,1,010201,AMAZONAS,BAGUA,BAGUA,29096,1,3.4369,...,16,86.4,0,1,0,0,0,0.5,0.866025,25
4,2018,1,1,010201,AMAZONAS,BAGUA,BAGUA,29096,1,3.4369,...,16,86.4,0,1,0,0,0,0.5,0.866025,47


In [42]:
resumen_columnas_modelo = pd.DataFrame({
    "columna": df_modelo.columns,
    "tipo_dato": df_modelo.dtypes.astype(str).values,
    "nulos": df_modelo.isna().sum().values,
    "porcentaje_nulos": (df_modelo.isna().mean() * 100).round(2).values,
    "valores_unicos": df_modelo.nunique(dropna=True).values
})

resumen_columnas_modelo

,columna,tipo_dato,nulos,porcentaje_nulos,valores_unicos
0,anio,int32,0,0.00,8
1,mes,int32,0,0.00,12
2,trimestre,int32,0,0.00,4
3,ubigeo,string,0,0.00,1833
4,departamento,string,0,0.00,26
5,provincia,string,0,0.00,195
6,distrito,string,0,0.00,1682
7,poblacion,int64,0,0.00,9061
8,n_comisarias_ref_2020,int64,0,0.00,11
9,comisarias_por_100k_ref_2020,float64,120625,15.61,6602


In [43]:
columnas_recomendadas = [
    "anio",
    "mes",
    "trimestre",
    "mes_sin",
    "mes_cos",

    "ubigeo",
    "departamento",
    "provincia",

    "poblacion",

    "n_comisarias_ref_2020",
    "comisarias_por_100k_ref_2020",
    "n_comisarias_rurales_ref_2020",
    "n_comisarias_sectoriales_ref_2020",
    "n_comisarias_zonales_ref_2020",
    "n_comisarias_tipo_A_ref_2020",
    "n_comisarias_tipo_B_ref_2020",
    "n_comisarias_tipo_C_ref_2020",
    "n_comisarias_tipo_D_ref_2020",
    "n_comisarias_tipo_E_ref_2020",
    "match_comisarias_ref_2020",

    "n_policias_ref_2025",
    "policias_por_100k_ref_2025",
    "pct_policias_femenino_ref_2025",
    "pct_policias_operativo_ref_2025",
    "match_policias",

    "dist_emergencia",
    "tipo_analisis",
    "nivel_detalle",

    "tipo",
    "sub_tipo",
    "modalidad",
    "categoria",

    "cantidad"
]

columnas_recomendadas = [
    col for col in columnas_recomendadas
    if col in df_modelo.columns
]

df_modelo_final = df_modelo[columnas_recomendadas].copy()

print("Tamaño de df_modelo:", df_modelo.shape)
print("Tamaño de df_modelo_final:", df_modelo_final.shape)

df_modelo_final.head()

Tamaño de df_modelo: (772656, 43)
Tamaño de df_modelo_final: (772656, 33)


,anio,mes,trimestre,mes_sin,mes_cos,ubigeo,departamento,provincia,poblacion,n_comisarias_ref_2020,...,pct_policias_operativo_ref_2025,match_policias,dist_emergencia,tipo_analisis,nivel_detalle,tipo,sub_tipo,modalidad,categoria,cantidad
0,2018,1,1,0.5,0.866025,010202,AMAZONAS,BAGUA,10422,2,...,100.0,SI,NO,PRINCIPALES_MODALIDADES,MODALIDAD_RESUMIDA,AGREGADO,AGREGADO,OTROS,OTROS,3
1,2018,1,1,0.5,0.866025,010202,AMAZONAS,BAGUA,10422,2,...,100.0,SI,NO,PRINCIPALES_TIPOS,TIPO,DELITOS CONTRA LA LIBERTAD,AGREGADO,AGREGADO,DELITOS CONTRA LA LIBERTAD,3
2,2018,1,1,0.5,0.866025,010201,AMAZONAS,BAGUA,29096,1,...,86.4,SI,NO,PRINCIPALES_MODALIDADES,MODALIDAD_RESUMIDA,AGREGADO,AGREGADO,ESTAFA,ESTAFA,2
3,2018,1,1,0.5,0.866025,010201,AMAZONAS,BAGUA,29096,1,...,86.4,SI,NO,PRINCIPALES_MODALIDADES,MODALIDAD_RESUMIDA,AGREGADO,AGREGADO,HURTO,HURTO,25
4,2018,1,1,0.5,0.866025,010201,AMAZONAS,BAGUA,29096,1,...,86.4,SI,NO,PRINCIPALES_MODALIDADES,MODALIDAD_RESUMIDA,AGREGADO,AGREGADO,OTROS,OTROS,47


In [44]:
resumen_columnas_modelo_final = pd.DataFrame({
    "columna": df_modelo_final.columns,
    "tipo_dato": df_modelo_final.dtypes.astype(str).values,
    "nulos": df_modelo_final.isna().sum().values,
    "porcentaje_nulos": (df_modelo_final.isna().mean() * 100).round(2).values,
    "valores_unicos": df_modelo_final.nunique(dropna=True).values
})

resumen_columnas_modelo_final

,columna,tipo_dato,nulos,porcentaje_nulos,valores_unicos
0,anio,int32,0,0.00,8
1,mes,int32,0,0.00,12
2,trimestre,int32,0,0.00,4
3,mes_sin,float64,0,0.00,11
4,mes_cos,float64,0,0.00,11
5,ubigeo,string,0,0.00,1833
6,departamento,string,0,0.00,26
7,provincia,string,0,0.00,195
8,poblacion,int64,0,0.00,9061
9,n_comisarias_ref_2020,int64,0,0.00,11


In [45]:
columnas_nulos_a_cero = [
    "comisarias_por_100k_ref_2020",
    "policias_por_100k_ref_2025",
    "pct_policias_femenino_ref_2025",
    "pct_policias_operativo_ref_2025"
]

columnas_nulos_a_cero = [
    col for col in columnas_nulos_a_cero
    if col in df_modelo_final.columns
]

for col in columnas_nulos_a_cero:
    df_modelo_final[col + "_es_nulo"] = df_modelo_final[col].isna().astype(int)
    df_modelo_final[col] = df_modelo_final[col].fillna(0)

df_modelo_final.head()

,anio,mes,trimestre,mes_sin,mes_cos,ubigeo,departamento,provincia,poblacion,n_comisarias_ref_2020,...,nivel_detalle,tipo,sub_tipo,modalidad,categoria,cantidad,comisarias_por_100k_ref_2020_es_nulo,policias_por_100k_ref_2025_es_nulo,pct_policias_femenino_ref_2025_es_nulo,pct_policias_operativo_ref_2025_es_nulo
0,2018,1,1,0.5,0.866025,010202,AMAZONAS,BAGUA,10422,2,...,MODALIDAD_RESUMIDA,AGREGADO,AGREGADO,OTROS,OTROS,3,0,0,0,0
1,2018,1,1,0.5,0.866025,010202,AMAZONAS,BAGUA,10422,2,...,TIPO,DELITOS CONTRA LA LIBERTAD,AGREGADO,AGREGADO,DELITOS CONTRA LA LIBERTAD,3,0,0,0,0
2,2018,1,1,0.5,0.866025,010201,AMAZONAS,BAGUA,29096,1,...,MODALIDAD_RESUMIDA,AGREGADO,AGREGADO,ESTAFA,ESTAFA,2,0,0,0,0
3,2018,1,1,0.5,0.866025,010201,AMAZONAS,BAGUA,29096,1,...,MODALIDAD_RESUMIDA,AGREGADO,AGREGADO,HURTO,HURTO,25,0,0,0,0
4,2018,1,1,0.5,0.866025,010201,AMAZONAS,BAGUA,29096,1,...,MODALIDAD_RESUMIDA,AGREGADO,AGREGADO,OTROS,OTROS,47,0,0,0,0


In [46]:
X = df_modelo_final.drop(columns=["cantidad"])
y = df_modelo_final["cantidad"]

columnas_numericas = X.select_dtypes(
    include=["int64", "int32", "float64", "float32"]
).columns.tolist()

columnas_categoricas = X.select_dtypes(
    include=["object", "string", "category", "bool"]
).columns.tolist()

print("Columnas numéricas:", len(columnas_numericas))
print(columnas_numericas)

print("\nColumnas categóricas:", len(columnas_categoricas))
print(columnas_categoricas)

Columnas numéricas: 24
['anio', 'mes', 'trimestre', 'mes_sin', 'mes_cos', 'poblacion', 'n_comisarias_ref_2020', 'comisarias_por_100k_ref_2020', 'n_comisarias_rurales_ref_2020', 'n_comisarias_sectoriales_ref_2020', 'n_comisarias_zonales_ref_2020', 'n_comisarias_tipo_A_ref_2020', 'n_comisarias_tipo_B_ref_2020', 'n_comisarias_tipo_C_ref_2020', 'n_comisarias_tipo_D_ref_2020', 'n_comisarias_tipo_E_ref_2020', 'n_policias_ref_2025', 'policias_por_100k_ref_2025', 'pct_policias_femenino_ref_2025', 'pct_policias_operativo_ref_2025', 'comisarias_por_100k_ref_2020_es_nulo', 'policias_por_100k_ref_2025_es_nulo', 'pct_policias_femenino_ref_2025_es_nulo', 'pct_policias_operativo_ref_2025_es_nulo']

Columnas categóricas: 12
['ubigeo', 'departamento', 'provincia', 'match_comisarias_ref_2020', 'match_policias', 'dist_emergencia', 'tipo_analisis', 'nivel_detalle', 'tipo', 'sub_tipo', 'modalidad', 'categoria']


In [47]:
df_modelo_limpio = df_modelo_final.copy()

for col in columnas_categoricas:
    df_modelo_limpio[col] = df_modelo_limpio[col].fillna("SIN_DATO")

print("Nulos restantes:", df_modelo_limpio.isna().sum().sum())

df_modelo_limpio.head()

Nulos restantes: 0


,anio,mes,trimestre,mes_sin,mes_cos,ubigeo,departamento,provincia,poblacion,n_comisarias_ref_2020,...,nivel_detalle,tipo,sub_tipo,modalidad,categoria,cantidad,comisarias_por_100k_ref_2020_es_nulo,policias_por_100k_ref_2025_es_nulo,pct_policias_femenino_ref_2025_es_nulo,pct_policias_operativo_ref_2025_es_nulo
0,2018,1,1,0.5,0.866025,010202,AMAZONAS,BAGUA,10422,2,...,MODALIDAD_RESUMIDA,AGREGADO,AGREGADO,OTROS,OTROS,3,0,0,0,0
1,2018,1,1,0.5,0.866025,010202,AMAZONAS,BAGUA,10422,2,...,TIPO,DELITOS CONTRA LA LIBERTAD,AGREGADO,AGREGADO,DELITOS CONTRA LA LIBERTAD,3,0,0,0,0
2,2018,1,1,0.5,0.866025,010201,AMAZONAS,BAGUA,29096,1,...,MODALIDAD_RESUMIDA,AGREGADO,AGREGADO,ESTAFA,ESTAFA,2,0,0,0,0
3,2018,1,1,0.5,0.866025,010201,AMAZONAS,BAGUA,29096,1,...,MODALIDAD_RESUMIDA,AGREGADO,AGREGADO,HURTO,HURTO,25,0,0,0,0
4,2018,1,1,0.5,0.866025,010201,AMAZONAS,BAGUA,29096,1,...,MODALIDAD_RESUMIDA,AGREGADO,AGREGADO,OTROS,OTROS,47,0,0,0,0


In [48]:
resumen_final_modelo = pd.DataFrame({
    "columna": df_modelo_limpio.columns,
    "tipo_dato": df_modelo_limpio.dtypes.astype(str).values,
    "nulos": df_modelo_limpio.isna().sum().values,
    "porcentaje_nulos": (df_modelo_limpio.isna().mean() * 100).round(2).values,
    "valores_unicos": df_modelo_limpio.nunique(dropna=True).values
})

resumen_final_modelo

,columna,tipo_dato,nulos,porcentaje_nulos,valores_unicos
0,anio,int32,0,0.0,8
1,mes,int32,0,0.0,12
2,trimestre,int32,0,0.0,4
3,mes_sin,float64,0,0.0,11
4,mes_cos,float64,0,0.0,11
5,ubigeo,string,0,0.0,1833
6,departamento,string,0,0.0,26
7,provincia,string,0,0.0,195
8,poblacion,int64,0,0.0,9061
9,n_comisarias_ref_2020,int64,0,0.0,11


In [49]:
df_modelo_limpio["fecha_aux"] = pd.to_datetime(
    df_modelo_limpio["anio"].astype(str) + "-" +
    df_modelo_limpio["mes"].astype(str).str.zfill(2) + "-01",
    errors="coerce"
)

df_modelo_limpio[["anio", "mes", "fecha_aux"]].head()

,anio,mes,fecha_aux
0,2018,1,2018-01-01
1,2018,1,2018-01-01
2,2018,1,2018-01-01
3,2018,1,2018-01-01
4,2018,1,2018-01-01


In [50]:
df_modelo_limpio = df_modelo_limpio.sort_values("fecha_aux").reset_index(drop=True)

fecha_corte = df_modelo_limpio["fecha_aux"].quantile(0.8)

train_pre = df_modelo_limpio[df_modelo_limpio["fecha_aux"] <= fecha_corte].copy()
test_pre = df_modelo_limpio[df_modelo_limpio["fecha_aux"] > fecha_corte].copy()

train_pre = train_pre.drop(columns=["fecha_aux"])
test_pre = test_pre.drop(columns=["fecha_aux"])

df_modelo_limpio_final = df_modelo_limpio.drop(columns=["fecha_aux"])

print("Fecha de corte:", fecha_corte)
print("Dataset completo:", df_modelo_limpio_final.shape)
print("Train:", train_pre.shape)
print("Test:", test_pre.shape)

Fecha de corte: 2024-12-01 00:00:00
Dataset completo: (772656, 37)
Train: (618273, 37)
Test: (154383, 37)


In [51]:
X_train = train_pre.drop(columns=["cantidad"])
y_train = train_pre["cantidad"]

X_test = test_pre.drop(columns=["cantidad"])
y_test = test_pre["cantidad"]

print("X_train:", X_train.shape)
print("y_train:", y_train.shape)
print("X_test:", X_test.shape)
print("y_test:", y_test.shape)

X_train: (618273, 36)
y_train: (618273,)
X_test: (154383, 36)
y_test: (154383,)


In [52]:
columnas_numericas_finales = X_train.select_dtypes(
    include=["int64", "int32", "float64", "float32"]
).columns.tolist()

columnas_categoricas_finales = X_train.select_dtypes(
    include=["object", "string", "category", "bool"]
).columns.tolist()

print("Columnas numéricas finales:", len(columnas_numericas_finales))
print(columnas_numericas_finales)

print("\nColumnas categóricas finales:", len(columnas_categoricas_finales))
print(columnas_categoricas_finales)

Columnas numéricas finales: 24
['anio', 'mes', 'trimestre', 'mes_sin', 'mes_cos', 'poblacion', 'n_comisarias_ref_2020', 'comisarias_por_100k_ref_2020', 'n_comisarias_rurales_ref_2020', 'n_comisarias_sectoriales_ref_2020', 'n_comisarias_zonales_ref_2020', 'n_comisarias_tipo_A_ref_2020', 'n_comisarias_tipo_B_ref_2020', 'n_comisarias_tipo_C_ref_2020', 'n_comisarias_tipo_D_ref_2020', 'n_comisarias_tipo_E_ref_2020', 'n_policias_ref_2025', 'policias_por_100k_ref_2025', 'pct_policias_femenino_ref_2025', 'pct_policias_operativo_ref_2025', 'comisarias_por_100k_ref_2020_es_nulo', 'policias_por_100k_ref_2025_es_nulo', 'pct_policias_femenino_ref_2025_es_nulo', 'pct_policias_operativo_ref_2025_es_nulo']

Columnas categóricas finales: 12
['ubigeo', 'departamento', 'provincia', 'match_comisarias_ref_2020', 'match_policias', 'dist_emergencia', 'tipo_analisis', 'nivel_detalle', 'tipo', 'sub_tipo', 'modalidad', 'categoria']


In [53]:
diccionario_columnas_modelo = pd.DataFrame({
    "columna": X_train.columns,
    "tipo_modelo": [
        "numerica" if col in columnas_numericas_finales else "categorica"
        for col in X_train.columns
    ],
    "es_variable_objetivo": "no"
})

fila_objetivo = pd.DataFrame({
    "columna": ["cantidad"],
    "tipo_modelo": ["objetivo"],
    "es_variable_objetivo": ["si"]
})

diccionario_columnas_modelo = pd.concat(
    [diccionario_columnas_modelo, fila_objetivo],
    ignore_index=True
)

diccionario_columnas_modelo

,columna,tipo_modelo,es_variable_objetivo
0,anio,numerica,no
1,mes,numerica,no
2,trimestre,numerica,no
3,mes_sin,numerica,no
4,mes_cos,numerica,no
5,ubigeo,categorica,no
6,departamento,categorica,no
7,provincia,categorica,no
8,poblacion,numerica,no
9,n_comisarias_ref_2020,numerica,no


In [54]:
resumen_preprocesamiento = pd.DataFrame({
    "elemento": [
        "filas_originales",
        "columnas_originales",
        "filas_dataset_modelo",
        "columnas_dataset_modelo",
        "filas_dataset_final",
        "columnas_dataset_final",
        "filas_train",
        "columnas_train",
        "filas_test",
        "columnas_test",
        "fecha_corte",
        "columnas_numericas",
        "columnas_categoricas",
        "nulos_dataset_final",
        "variable_objetivo"
    ],
    "valor": [
        df.shape[0],
        df.shape[1],
        df_modelo.shape[0],
        df_modelo.shape[1],
        df_modelo_limpio_final.shape[0],
        df_modelo_limpio_final.shape[1],
        train_pre.shape[0],
        train_pre.shape[1],
        test_pre.shape[0],
        test_pre.shape[1],
        fecha_corte,
        len(columnas_numericas_finales),
        len(columnas_categoricas_finales),
        df_modelo_limpio_final.isna().sum().sum(),
        "cantidad"
    ]
})

resumen_preprocesamiento

,elemento,valor
0,filas_originales,772656
1,columnas_originales,56
2,filas_dataset_modelo,772656
3,columnas_dataset_modelo,43
4,filas_dataset_final,772656
5,columnas_dataset_final,37
6,filas_train,618273
7,columnas_train,37
8,filas_test,154383
9,columnas_test,37


In [55]:
from pathlib import Path

carpeta_salida = Path(
    r"../Data/salida_preprocesamiento"
)

carpeta_salida.mkdir(parents=True, exist_ok=True)

ruta_dataset = carpeta_salida / "dataset_preprocesado_modelo_v5.csv"
ruta_train = carpeta_salida / "train_preprocesado_v5.csv"
ruta_test = carpeta_salida / "test_preprocesado_v5.csv"
ruta_x_train = carpeta_salida / "X_train_preprocesado_v5.csv"
ruta_y_train = carpeta_salida / "y_train_preprocesado_v5.csv"
ruta_x_test = carpeta_salida / "X_test_preprocesado_v5.csv"
ruta_y_test = carpeta_salida / "y_test_preprocesado_v5.csv"
ruta_columnas_iniciales = carpeta_salida / "resumen_columnas_inicial_v5.csv"
ruta_columnas_modelo = carpeta_salida / "columnas_modelo_v5.csv"
ruta_columnas_finales = carpeta_salida / "columnas_modelo_final_v5.csv"
ruta_diccionario = carpeta_salida / "diccionario_columnas_modelo_v5.csv"
ruta_resumen = carpeta_salida / "resumen_preprocesamiento_v5.csv"

df_modelo_limpio_final.to_csv(ruta_dataset, index=False, encoding="utf-8-sig")
train_pre.to_csv(ruta_train, index=False, encoding="utf-8-sig")
test_pre.to_csv(ruta_test, index=False, encoding="utf-8-sig")
X_train.to_csv(ruta_x_train, index=False, encoding="utf-8-sig")
y_train.to_csv(ruta_y_train, index=False, encoding="utf-8-sig")
X_test.to_csv(ruta_x_test, index=False, encoding="utf-8-sig")
y_test.to_csv(ruta_y_test, index=False, encoding="utf-8-sig")

resumen_inicial.to_csv(ruta_columnas_iniciales, index=False, encoding="utf-8-sig")
resumen_columnas_modelo.to_csv(ruta_columnas_modelo, index=False, encoding="utf-8-sig")
resumen_final_modelo.to_csv(ruta_columnas_finales, index=False, encoding="utf-8-sig")
diccionario_columnas_modelo.to_csv(ruta_diccionario, index=False, encoding="utf-8-sig")
resumen_preprocesamiento.to_csv(ruta_resumen, index=False, encoding="utf-8-sig")

print("Archivos guardados en:")
print(carpeta_salida)

Archivos guardados en:
..\Data\salida_preprocesamiento


In [56]:
archivos_generados = pd.DataFrame({
    "archivo": [
        ruta_dataset.name,
        ruta_train.name,
        ruta_test.name,
        ruta_x_train.name,
        ruta_y_train.name,
        ruta_x_test.name,
        ruta_y_test.name,
        ruta_columnas_iniciales.name,
        ruta_columnas_modelo.name,
        ruta_columnas_finales.name,
        ruta_diccionario.name,
        ruta_resumen.name
    ],
    "existe": [
        ruta_dataset.exists(),
        ruta_train.exists(),
        ruta_test.exists(),
        ruta_x_train.exists(),
        ruta_y_train.exists(),
        ruta_x_test.exists(),
        ruta_y_test.exists(),
        ruta_columnas_iniciales.exists(),
        ruta_columnas_modelo.exists(),
        ruta_columnas_finales.exists(),
        ruta_diccionario.exists(),
        ruta_resumen.exists()
    ]
})

archivos_generados

,archivo,existe
0,dataset_preprocesado_modelo_v5.csv,True
1,train_preprocesado_v5.csv,True
2,test_preprocesado_v5.csv,True
3,X_train_preprocesado_v5.csv,True
4,y_train_preprocesado_v5.csv,True
5,X_test_preprocesado_v5.csv,True
6,y_test_preprocesado_v5.csv,True
7,resumen_columnas_inicial_v5.csv,True
8,columnas_modelo_v5.csv,True
9,columnas_modelo_final_v5.csv,True
